In [5]:
# -*- coding: utf-8 -*-
"""
Threshold grid runner for MODIS vs Sentinel-2 validation (FINAL, fixed).

Runs these 9 (MODIS,S2) NDVI threshold pairs:
[0.20,0.20],[0.20,0.15],[0.20,0.10],
[0.15,0.20],[0.15,0.15],[0.15,0.10],
[0.10,0.20],[0.10,0.15],[0.10,0.10]

For each pair and each year:
  1) Sample up to CANDIDATES_PER_YEAR MODIS pixels with MODIS NDVI >= MODIS_THR
  2) EE: within [Sep 23, year]..[Mar 21, year+1], find S2 max NDVI and count 10 m pixels with NDVI > S2_THR inside a BUFFER_M circle
  3) Keep only green_px_count>0 and MODIS NDVI within a MODIS range:
       - If USE_NDVI_BANDS=True: [MODIS_THR, MODIS_THR+BAND_WIDTH)
       - Else:                    [MODIS_THR, 1.0)
  4) Cap to TARGET_PER_YEAR (using a seed that depends on the pair) and save CSV

Output CSV name pattern:
  <OUT_DIR>/S2_green_pixel_count_2018_2021_MODIS{m:.2f}_S2{s:.2f}.csv

Dependencies: rasterio, numpy, pandas, pyproj, tqdm, earthengine-api
"""

from pathlib import Path
import random
import numpy as np
import pandas as pd
import rasterio
import pyproj
from tqdm import tqdm
import ee
import os

# -------------------- USER CONFIG --------------------
TIF_PATHS = {
    2018: r"G:\Hangkai\Anttarctic Vegetation Dynamic\Version_2_data\Antractic_Max_NDVI_every_year\Different_Threshold\MaxNDVI_00_2018-0000000000-0000000000.tif",
    2019: r"G:\Hangkai\Anttarctic Vegetation Dynamic\Version_2_data\Antractic_Max_NDVI_every_year\Different_Threshold\MaxNDVI_00_2019-0000000000-0000000000.tif",
    2020: r"G:\Hangkai\Anttarctic Vegetation Dynamic\Version_2_data\Antractic_Max_NDVI_every_year\Different_Threshold\MaxNDVI_00_2020-0000000000-0000000000.tif",
    2021: r"G:\Hangkai\Anttarctic Vegetation Dynamic\Version_2_data\Antractic_Max_NDVI_every_year\Different_Threshold\MaxNDVI_00_2021-0000000000-0000000000.tif",
}

# Use all 9 pairs; comment lines if you want fewer
THRESHOLD_PAIRS = [
    (0.20, 0.20), (0.20, 0.15), (0.20, 0.10),
    (0.15, 0.20), (0.15, 0.15), (0.15, 0.10),
    (0.10, 0.20), (0.10, 0.15), (0.10, 0.10),
]

TARGET_PER_YEAR     = 250
CANDIDATES_PER_YEAR = 10000 # add more if accepted points are less than 250
BUFFER_M            = 353
CLOUDY_PCT_MAX      = 10
SZA_MAX_DEG         = 70
SEED                = 42
OUT_DIR             = r"G:\Hangkai\Anttarctic Vegetation Dynamic\Version_2_data\threshold_grid_outputs_353"

# ---- make MODIS rows disjoint by NDVI bands ----
USE_NDVI_BANDS = True
BAND_WIDTH     = 0.05

os.makedirs(OUT_DIR, exist_ok=True)

# ---- deterministic seed helper that varies with (year, modis_thr, s2_thr, stage) ----
def _seed(*parts):
    """
    Build a deterministic integer seed from components.
    Use: _seed(year-2000, modis_thr, s2_thr, stage_id)
    """
    acc = SEED
    for p in parts:
        if isinstance(p, float):
            acc += int(round(p * 1000))
        else:
            acc += int(p)
    return int(acc)

# -------------------- EE INIT --------------------
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

# -------------------- EE HELPERS --------------------
def mask_s2_sr(image: ee.Image) -> ee.Image:
    """Keep SCL vegetation(4), bare soil(5), water(6), snow/ice(11)."""
    scl = image.select('SCL')
    good = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(11))
    return image.updateMask(good)

def add_s2_ndvi(image: ee.Image) -> ee.Image:
    return image.addBands(image.normalizedDifference(['B8','B4']).rename('NDVI'))

def s2_collection_for_window(point_geom: ee.Geometry, year: int) -> ee.ImageCollection:
    start_date = ee.Date.fromYMD(year, 9, 23)
    end_date   = ee.Date.fromYMD(year + 1, 3, 21)
    return (ee.ImageCollection("COPERNICUS/S2_SR")
            .filterBounds(point_geom)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUDY_PCT_MAX))
            .filter(ee.Filter.lt('MEAN_SOLAR_ZENITH_ANGLE', SZA_MAX_DEG))
            .map(mask_s2_sr)
            .map(add_s2_ndvi))

def _ee_map_per_feat(year: int, s2_thr: float):
    """Server-side mapper with custom S2 NDVI threshold."""
    def per_feat(feat):
        geom = feat.geometry()
        s2   = s2_collection_for_window(geom, year)
        has_any = s2.size().gt(0)

        def when_empty():
            return feat.set({'s2_has_data': False, 'green_px_count': 0})

        def when_nonempty():
            # seasonal MAX NDVI, then threshold
            max_ndvi = s2.select('NDVI').max()
            green    = max_ndvi.gt(s2_thr).rename('green').selfMask()
            green_count = green.reduceRegion(
                reducer=ee.Reducer.count(), geometry=geom, scale=10,
                maxPixels=1e9, bestEffort=True, tileScale=2
            ).get('green')

            green_count_safe = ee.Number(
                ee.Algorithms.If(ee.Algorithms.IsEqual(green_count, None), 0, green_count)
            ).toInt()

            return feat.set({'s2_has_data': True, 'green_px_count': green_count_safe})

        return ee.Algorithms.If(has_any, when_nonempty(), when_empty())
    return per_feat

# -------------------- RASTER HELPERS --------------------
def _auto_scale_modis_ndvi(arr: np.ndarray) -> np.ndarray:
    """Auto-scale common MODIS NDVI integers to [-1,1] by 0.0001 if needed."""
    if np.issubdtype(arr.dtype, np.integer) or (np.nanmax(arr) > 1.5):
        arr = arr.astype(np.float32) * 0.0001
    return np.clip(arr, -1.0, 1.0)

def sample_candidates_from_tif(tif_path: str,
                               modis_thr: float,
                               max_candidates: int,
                               seed: int = 42) -> pd.DataFrame:
    """Return DataFrame with idx,row,col,ndvi,lon,lat for MODIS NDVI >= modis_thr."""
    rng = np.random.default_rng(seed)
    with rasterio.open(tif_path) as src:
        ndvi = _auto_scale_modis_ndvi(src.read(1))
        ndvi = np.clip(ndvi, -1.0, 1.0)

        mask = np.isfinite(ndvi) & (ndvi >= modis_thr) & (ndvi <= 1.0)
        rows, cols = np.where(mask)
        if len(rows) == 0:
            return pd.DataFrame(columns=['idx','row','col','ndvi','lon','lat'])

        idx_all = np.arange(len(rows))
        rng.shuffle(idx_all)
        idx_sel = idx_all[:min(max_candidates, len(idx_all))]
        rows = rows[idx_sel]
        cols = cols[idx_sel]
        vals = ndvi[rows, cols]

        xs, ys = rasterio.transform.xy(src.transform, rows, cols, offset='center')
        xs = np.array(xs); ys = np.array(ys)
        transformer = pyproj.Transformer.from_crs(src.crs, "EPSG:4326", always_xy=True)
        lons, lats = transformer.transform(xs, ys)

        return pd.DataFrame({
            'idx': np.arange(len(rows), dtype=int),
            'row': rows.astype(int),
            'col': cols.astype(int),
            'ndvi': vals.astype(float),
            'lon': lons.astype(float),
            'lat': lats.astype(float)
        })

def ee_eval_points(points_df: pd.DataFrame, year: int, buffer_m: int, s2_thr: float, chunk_size: int = 150) -> pd.DataFrame:
    """Compute s2_has_data & green_px_count for each point at S2 threshold s2_thr."""
    out_rows = []
    per_feat = _ee_map_per_feat(year=year, s2_thr=s2_thr)

    for i in tqdm(range(0, len(points_df), chunk_size), desc=f"EE eval year {year}, S2>{s2_thr}"):
        chunk = points_df.iloc[i:i+chunk_size].copy()

        feats = []
        for _, r in chunk.iterrows():
            pt = ee.Geometry.Point([float(r['lon']), float(r['lat'])]).buffer(buffer_m)
            feats.append(ee.Feature(pt, {
                'idx' : int(r['idx']),
                'row' : int(r['row']),
                'col' : int(r['col']),
                'lon' : float(r['lon']),
                'lat' : float(r['lat']),
                'modis_ndvi' : float(r['ndvi'])
            }))

        fc = ee.FeatureCollection(feats).map(per_feat)

        try:
            data = fc.getInfo()['features']
        except Exception as e:
            print(f"Warning: EE chunk failed ({e}); skipping {len(chunk)} points.")
            continue

        for f in data:
            p = f['properties']
            out_rows.append({
                'idx': int(p['idx']),
                'row': int(p['row']),
                'col': int(p['col']),
                'lon': float(p['lon']),
                'lat': float(p['lat']),
                'modis_ndvi': float(p['modis_ndvi']),
                's2_has_data': bool(p['s2_has_data']),
                'green_px_count': int(p['green_px_count']),
            })

    cols = ['idx','row','col','lon','lat','modis_ndvi','s2_has_data','green_px_count']
    if not out_rows:
        return pd.DataFrame(columns=cols)
    df = pd.DataFrame(out_rows, columns=cols)
    df['modis_ndvi'] = df['modis_ndvi'].clip(0, 1)
    return df

# -------------------- MAIN --------------------
def run_pair(modis_thr: float, s2_thr: float):
    all_year_rows = []

    for year, tif_path in TIF_PATHS.items():
        print(f"\n=== Year {year} | MODIS≥{modis_thr:.2f} | S2>{s2_thr:.2f} ===")

        # ---- FIX 1: candidate sampling seed depends on year + modis_thr
        seed_cand = _seed(year - 2000, modis_thr, 1)  # stage=1
        cand_df = sample_candidates_from_tif(
            tif_path, modis_thr=modis_thr,
            max_candidates=CANDIDATES_PER_YEAR,
            seed=seed_cand
        )
        if cand_df.empty:
            print("No MODIS candidates; skipping year.")
            continue

        eval_df = ee_eval_points(cand_df, year=year, buffer_m=BUFFER_M, s2_thr=s2_thr, chunk_size=150)
        if eval_df.empty:
            print("All EE chunks failed or no valid S2 data; skipping year.")
            continue

        # ---- FIX 2: disjoint MODIS rows by NDVI bands (optional)
        if USE_NDVI_BANDS:
            upper = min(1.0, modis_thr + BAND_WIDTH)
            ok = eval_df[
                (eval_df['green_px_count'] > 0) &
                (eval_df['modis_ndvi'] >= modis_thr) &
                (eval_df['modis_ndvi'] <  upper)
            ].copy()
        else:
            ok = eval_df[
                (eval_df['green_px_count'] > 0) &
                (eval_df['modis_ndvi'] >= modis_thr) &
                (eval_df['modis_ndvi'] < 1.0)
            ].copy()

        if ok.empty:
            print("No points satisfy green_px_count>0 with MODIS NDVI in range.")
            continue

        # ---- FIX 3: final cap seed depends on (year, modis_thr, s2_thr)
        if len(ok) > TARGET_PER_YEAR:
            seed_cap = _seed(year - 2000, modis_thr, s2_thr, 2)  # stage=2
            ok = ok.sample(TARGET_PER_YEAR, random_state=seed_cap)

        ok['year'] = year
        ok['modis_thr'] = float(modis_thr)
        ok['s2_thr'] = float(s2_thr)
        ok = ok[['year','row','col','lon','lat','modis_ndvi','s2_has_data','green_px_count','modis_thr','s2_thr']]
        print(f"Accepted {len(ok)} pixels for {year} (target {TARGET_PER_YEAR}).")
        all_year_rows.append(ok)

    if not all_year_rows:
        print("No data collected for this pair.")
        return None

    out_df = pd.concat(all_year_rows, ignore_index=True).sort_values(['year']).reset_index(drop=True)
    out_name = f"S2_green_pixel_count_2018_2021_MODIS{modis_thr:.2f}_S2{s2_thr:.2f}.csv"
    out_path = str(Path(OUT_DIR) / out_name)
    out_df.to_csv(out_path, index=False, float_format="%.6f")
    print(f"\nSaved: {out_path}")
    return out_path

if __name__ == "__main__":
    for (m_thr, s_thr) in THRESHOLD_PAIRS:
        run_pair(m_thr, s_thr)


=== Year 2018 | MODIS≥0.20 | S2>0.20 ===


EE eval year 2018, S2>0.2: 100%|███████████████████████████████████████████████████████| 67/67 [08:28<00:00,  7.59s/it]


Accepted 250 pixels for 2018 (target 250).

=== Year 2019 | MODIS≥0.20 | S2>0.20 ===


EE eval year 2019, S2>0.2: 100%|███████████████████████████████████████████████████████| 67/67 [14:51<00:00, 13.30s/it]


Accepted 250 pixels for 2019 (target 250).

=== Year 2020 | MODIS≥0.20 | S2>0.20 ===


...

Accepted 250 pixels for 2019 (target 250).

=== Year 2020 | MODIS≥0.10 | S2>0.10 ===


EE eval year 2020, S2>0.1: 100%|███████████████████████████████████████████████████████| 67/67 [11:16<00:00, 10.10s/it]


Accepted 250 pixels for 2020 (target 250).

=== Year 2021 | MODIS≥0.10 | S2>0.10 ===


EE eval year 2021, S2>0.1: 100%|███████████████████████████████████████████████████████| 67/67 [17:17<00:00, 15.49s/it]

Accepted 250 pixels for 2021 (target 250).

Saved: G:\Hangkai\Anttarctic Vegetation Dynamic\Version_2_data\threshold_grid_outputs_353\S2_green_pixel_count_2018_2021_MODIS0.10_S20.10.csv
